# Task labor simulator

Estimate the **total labor (seconds) of one pick task** with the *real* pick-cost model, so you can
sanity-check that a task's labor estimate is sound.

A task = **N pick stops** along a within-aisle route. Each stop picks a tunable SKU (weight, volume,
quantity) at a bin height; between stops the picker travels a grounded distance.

```
total    = handling + travel + cart
  handling = N * M(y) * (intercept + qty*(pw*fn_w(weight) + pv*fn_v(volume)))
  travel   = N * manhattan_ft * ((1-vert_frac)/walk_ft_s + vert_frac/lift_ft_s)   # time = dist/speed
  cart     = cart_swap * swaps        (swaps from total picked volume vs cart capacity)
```

Change SKU params/quantity, pick the **base-function family** (`log` / `linear` / `sqrt` / `pow`) for
each handling term, and **ground travel** by equipment **speeds in ft/s** (horizontal walk, vertical
lift). The model matches the real `Pick._pick_time` when both base functions are `log` (asserted at
load). Travel speeds are ft/s and bin positions are inches, so the config emits `x_speed`/`y_speed`
directly in ft/s.

In [ ]:
import os, sys, math, pathlib
from math import ceil
import numpy as np
import matplotlib.pyplot as plt

# find repo root (dir containing Warehouse/) walking up from cwd
p = pathlib.Path.cwd()
while p != p.parent and not (p / 'Warehouse').exists():
    p = p.parent
ROOT = str(p)
for sub in ('Warehouse', 'Optimization'):
    d = os.path.join(ROOT, sub)
    if d not in sys.path:
        sys.path.insert(0, d)

# the REAL sim cost primitives (single source of truth)
from Pick import (PickConfig, _pick_time, DEFAULT_HEIGHT_BRACKETS,
                  height_multiplier, _CART_CAPACITY)
try:
    import ipywidgets as W
    _HAVE_WIDGETS = True
except Exception:
    _HAVE_WIDGETS = False
print('repo root:', ROOT, '| widgets:', _HAVE_WIDGETS, '| cart capacity:', _CART_CAPACITY)

In [ ]:
# Carton archetypes (weight, volume) spanning the distribution — slider defaults / reference points.
ITEMS = {
    'light':   (3,    27),
    'typical': (20,   27_000),
    'heavy':   (55,   110_000),
    'bulky':   (15,   110_000),
}
COL_STEP = 48                    # pallet column physical step (Aisle_Dimensions)
LVL_STEP = 48                    # level (height) physical step

In [ ]:
# ── composable cost model (prototype mirroring planned cost_model.LaborModel) ──
TRANSFORMS = {
    'log':    lambda v: math.log(max(v, 1.0)),   # natural log (base e)
    'linear': lambda v: float(v),
    'sqrt':   lambda v: math.sqrt(max(v, 0.0)),
}

def resolve_transform(spec):
    """'name' or 'name:param' -> callable. 'pow:0.5'->v**0.5 ; 'log'->ln ; 'log:10'->log base 10."""
    if ':' in spec:
        name, p = spec.split(':', 1)
        if name == 'pow':
            p = float(p)
            return lambda v: max(v, 0.0) ** p
        if name == 'log':
            if p == 'e':
                return TRANSFORMS['log']
            lnb = math.log(float(p))            # log_b(x) = ln(x) / ln(b)
            return lambda v: math.log(max(v, 1.0)) / lnb
        raise ValueError(f'unknown parametric transform {name!r}')
    return TRANSFORMS[spec]

def term_spec(fam, pow_p=0.5, log_base='e'):
    """Compose a family + its parameter into a transform spec the model accepts."""
    if fam == 'pow':  return f'pow:{pow_p:g}'
    if fam == 'log':  return 'log' if log_base == 'e' else f'log:{log_base}'
    return fam

def handle_var_fn(weight, volume, wc, vc, wfn, vfn):
    """Per-unit weight/volume handling term: wc*fn_w(weight) + vc*fn_v(volume)."""
    return wc * resolve_transform(wfn)(weight) + vc * resolve_transform(vfn)(volume)

def _fn_symbol(spec, var='w'):
    """Symbolic string for a spec: 'log'->'ln(w)', 'log:10'->'log10(w)', 'pow:0.5'->'w^0.5'."""
    if ':' in spec:
        name, p = spec.split(':', 1)
        if name == 'pow':  return f'{var}^{p}'
        if name == 'log':  return f'ln({var})' if p == 'e' else f'log{p}({var})'
        return f'{name}:{p}({var})'
    if spec == 'log':    return f'ln({var})'
    if spec == 'linear': return var
    if spec == 'sqrt':   return f'sqrt({var})'
    return f'{spec}({var})'

def _fmt_brackets(bs):
    """Repr height brackets with float('inf') so the printed config is valid Python."""
    parts = ["(float('inf'), {!r})".format(m) if math.isinf(t) else "({!r}, {!r})".format(t, m)
             for t, m in bs]
    return "(" + ", ".join(parts) + ")"

def _inv(speed_ft_per_sec):
    """1/speed, guarding speed<=0 -> inf (matches cost_model.sec_per_inch's guard)."""
    return 1.0 / speed_ft_per_sec if speed_ft_per_sec > 0 else float('inf')

def task_labor(weight, volume, qty, n_picks, bin_level, manhattan_ft, frac_vertical,
               intercept, pick_weight_coef, pick_volume_coef, weight_fn, volume_fn,
               cart_swap, walk_ft_per_sec, lift_ft_per_sec):
    """Total labor (seconds) for a task of `n_picks` stops of one SKU along a within-aisle route.
    Travel SPEEDS are in ft/s and leg distance in feet, so travel time = distance / speed
    (matching the real model: positions are inches, speed ft/s).  Returns the full breakdown."""
    hv       = handle_var_fn(weight, volume, pick_weight_coef, pick_volume_coef, weight_fn, volume_fn)
    M        = height_multiplier(DEFAULT_HEIGHT_BRACKETS, bin_level * LVL_STEP)
    per_pick = M * (intercept + qty * hv)
    handling = n_picks * per_pick
    per_leg  = manhattan_ft * ((1 - frac_vertical) * _inv(walk_ft_per_sec)
                               + frac_vertical * _inv(lift_ft_per_sec))
    travel   = n_picks * per_leg
    swaps    = max(0, ceil(n_picks * qty * volume / _CART_CAPACITY) - 1)
    cart     = swaps * cart_swap
    total    = handling + travel + cart
    return dict(hv=hv, M=M, per_pick=per_pick, handling=handling, per_leg=per_leg,
                travel=travel, swaps=swaps, cart=cart, total=total)

# Soundness: at fn=log, ground level, no swap, single-pick handling must equal the REAL
# Pick._pick_time (what the sim charges) — so the estimate is grounded, not a parallel formula.
def _check_against_real_pick_time():
    w, v, q = 20, 27_000, 3
    cfg = PickConfig(pick_intercept=15.0, pick_weight_coef=0.7, pick_volume_coef=0.09,
                     cart_swap_coef=0.0)
    r = task_labor(w, v, q, n_picks=1, bin_level=0, manhattan_ft=0.0, frac_vertical=0.0,
                   intercept=15.0, pick_weight_coef=0.7, pick_volume_coef=0.09,
                   weight_fn='log', volume_fn='log', cart_swap=0.0,
                   walk_ft_per_sec=4.0, lift_ft_per_sec=2.0)
    expect = _pick_time(cfg, w, v, q, False)
    assert abs(r['handling'] - expect) < 1e-9, (r['handling'], expect)
    return r['handling'], expect

_h, _e = _check_against_real_pick_time()
print(f'soundness OK: task_labor single-pick handling {_h:.4f} == _pick_time {_e:.4f}  (fn=log)')

In [ ]:
# ── Task labor simulator ──────────────────────────────────────────────────────
_FAMILIES = ['log', 'linear', 'sqrt', 'pow']   # 'pow' uses *_pow_p; 'log' uses *_log_base

def simulate(weight=20, volume=27_000, qty=2, n_picks=15, bin_level=0,
             manhattan_ft=32.0, frac_vertical=0.3,
             intercept=15.0, pick_weight_coef=0.7, pick_volume_coef=0.09,
             weight_fn='log', weight_pow_p=0.5, weight_log_base='e',
             volume_fn='log', volume_pow_p=0.5, volume_log_base='e',
             cart_swap=300.0, walk_ft_per_sec=4.0, lift_ft_per_sec=2.0,
             config_name='calibrated'):
    wsel = term_spec(weight_fn, weight_pow_p, weight_log_base)
    vsel = term_spec(volume_fn, volume_pow_p, volume_log_base)
    r = task_labor(weight, volume, qty, n_picks, bin_level, manhattan_ft, frac_vertical,
                   intercept, pick_weight_coef, pick_volume_coef, wsel, vsel,
                   cart_swap, walk_ft_per_sec, lift_ft_per_sec)
    H, T, C, tot = r['handling'], r['travel'], r['cart'], r['total']

    # one stacked breakdown bar; total in the title
    fig, ax = plt.subplots(figsize=(8, 1.7))
    left = 0.0
    for label, val, color in (('handling', H, '#dd8452'), ('travel', T, '#4c72b0'), ('cart', C, '#55a868')):
        if val > 0:
            ax.barh([0], [val], left=[left], color=color, label=f'{label} {val:.0f}s')
            left += val
    ax.set_yticks([]); ax.set_xlabel('task labor (s)')
    ax.set_title(f'TOTAL task labor = {tot:.0f} s  ({tot/60:.1f} min)   |   {n_picks} picks, qty {qty}')
    ax.legend(fontsize=8, loc='center right'); plt.tight_layout(); plt.show()

    sh = lambda x: (x / tot * 100 if tot else 0.0)
    print(f'shares: handling {sh(H):.0f}%  |  travel {sh(T):.0f}%  |  cart {sh(C):.0f}%   '
          f'(M={r["M"]:g}, cart swaps={r["swaps"]})')
    print()
    # ── how the estimate works (symbolic -> with current values) ──
    print('Formula (symbolic -> with current values):')
    print('  total    = handling + travel + cart')
    print('  handling = N * M(y) * (intercept + qty*(pw*fn_w(w) + pv*fn_v(v)))')
    print('  travel   = N * manhattan_ft * ((1-vf)/walk_ft_s + vf/lift_ft_s)   [dist/speed]')
    print('  cart     = cart_swap * max(0, ceil(N*qty*volume / cart_cap) - 1)')
    print(f'  handling = {n_picks} * {r["M"]:g} * ({intercept:g} + {qty:g}*({pick_weight_coef:.4g}*{_fn_symbol(wsel,"w")} '
          f'+ {pick_volume_coef:.4g}*{_fn_symbol(vsel,"v")})) = {H:.0f}')
    print(f'  travel   = {n_picks} * {manhattan_ft:g} * ((1-{frac_vertical:g})/{walk_ft_per_sec:g} '
          f'+ {frac_vertical:g}/{lift_ft_per_sec:g}) = {T:.0f}')
    print(f'  cart     = {cart_swap:g} * {r["swaps"]} = {C:.0f}')
    print(f'  total    = {H:.0f} + {T:.0f} + {C:.0f} = {tot:.0f} s')
    print()
    print('# --- paste into REGRESSION_CONFIGS (Optimization/run_simulation.py) ---')
    print('    {')
    print(f"        'name'            : {config_name!r},")
    print(f"        'pick_intercept'  : {intercept:.6g},")
    print(f"        'pick_weight_coef': {pick_weight_coef:.6g},")
    print(f"        'pick_weight_fn'  : {wsel!r},")
    print(f"        'pick_volume_coef': {pick_volume_coef:.6g},")
    print(f"        'pick_volume_fn'  : {vsel!r},")
    print(f"        'cart_swap_coef'  : {cart_swap:.6g},")
    print(f"        'x_speed'         : {walk_ft_per_sec:.6g},    # ft/s")
    print(f"        'y_speed'         : {lift_ft_per_sec:.6g},    # ft/s")
    print(f"        'height_brackets' : {_fmt_brackets(DEFAULT_HEIGHT_BRACKETS)},")
    print('    },')

if _HAVE_WIDGETS:
    W.interact(
        simulate,
        weight=W.FloatSlider(value=20, min=1, max=60, step=1, description='weight'),
        volume=W.FloatSlider(value=27_000, min=27, max=110_000, step=1000, description='volume'),
        qty=W.IntSlider(value=2, min=1, max=12, description='qty/pick'),
        n_picks=W.IntSlider(value=15, min=1, max=100, description='picks'),
        bin_level=W.IntSlider(value=0, min=0, max=8, description='bin level'),
        manhattan_ft=W.FloatSlider(value=32.0, min=0, max=200, step=2, description='manhattan ft'),
        frac_vertical=W.FloatSlider(value=0.3, min=0, max=1, step=0.05, description='vert frac'),
        intercept=W.FloatSlider(value=15.0, min=0, max=60, step=1, description='intercept'),
        pick_weight_coef=W.FloatSlider(value=0.7, min=0, max=6, step=0.02, description='weight coef'),
        pick_volume_coef=W.FloatSlider(value=0.09, min=0, max=1.5, step=0.005, description='volume coef'),
        weight_fn=W.Dropdown(options=_FAMILIES, value='log', description='weight fn'),
        weight_pow_p=W.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description='weight pow p'),
        weight_log_base=W.Dropdown(options=['e', '2', '10'], value='e', description='weight log base'),
        volume_fn=W.Dropdown(options=_FAMILIES, value='log', description='volume fn'),
        volume_pow_p=W.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description='volume pow p'),
        volume_log_base=W.Dropdown(options=['e', '2', '10'], value='e', description='volume log base'),
        cart_swap=W.FloatSlider(value=300.0, min=0, max=600, step=10, description='cart swap'),
        walk_ft_per_sec=W.FloatSlider(value=4.0, min=0.5, max=12, step=0.1, description='walk ft/s'),
        lift_ft_per_sec=W.FloatSlider(value=2.0, min=0.5, max=12, step=0.1, description='lift ft/s'),
        config_name=W.Text(value='calibrated', description='config name'),
    )
else:
    simulate()

## Ground the geometry to a real run

The simulator is only honest if its task geometry (`n_picks`, `manhattan_per_leg`, `frac_vertical`)
reflects reality. `actual_travel_share(run_dir)` reads the **realised** handling/travel split from a
completed run's `stats_summary.csv` — match the simulator's split to it, then trust the totals.

In [ ]:
# ── Grounding helper: realised travel/handling split from a completed run ──────
def actual_travel_share(run_dir, config='calibrated'):
    """Read per-strategy travel_time/handling_time means from a run's stats_summary.csv
    (run_analysis must have produced stats/) and report the realised travel share.  Match the
    simulator's n_picks / manhattan / vert-frac so its split reproduces this before trusting totals."""
    import glob, csv
    tt, ht = [], []
    for f in glob.glob(os.path.join(run_dir, '*', config, 'stats', 'stats_summary.csv')):
        with open(f, encoding='utf-8') as fh:
            for r in csv.DictReader(fh):
                if r['metric'] == 'travel_time':   tt.append(float(r['mean']))
                if r['metric'] == 'handling_time': ht.append(float(r['mean']))
    if not tt:
        print('no stats_summary.csv under', run_dir, '(run run_analysis with stats first)')
        return None
    T, H = sum(tt) / len(tt), sum(ht) / len(ht)
    share = T / (T + H)
    print(f'{config}: actual travel share = {share:.1%}   (travel {T:,.0f} / handling {H:,.0f}, n={len(tt)})')
    return share

# Example (edit path):
# actual_travel_share(r'H:\Data\Inventory_Optimizer_Data\Optimization_Outputs\comparison_20260621_192615')